# 1. 라이브러리

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score

# 2. 데이터셋 로드

In [ ]:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/car/car.data"
columns = ['price', 'maint', 'doors', 'persons', 'lug_capacity', 'safety', 'output']
df = pd.read_csv(url, header=None, names=columns)

df

,price,maint,doors,persons,lug_capacity,safety,output
0,vhigh,vhigh,2,2,small,low,unacc
1,vhigh,vhigh,2,2,small,med,unacc
2,vhigh,vhigh,2,2,small,high,unacc
3,vhigh,vhigh,2,2,med,low,unacc
4,vhigh,vhigh,2,2,med,med,unacc
...,...,...,...,...,...,...,...
1723,low,low,5more,more,med,med,good
1724,low,low,5more,more,med,high,vgood
1725,low,low,5more,more,big,low,unacc
1726,low,low,5more,more,big,med,good


# 3. EDA

In [ ]:
label_encoders = {}
for col in columns:
    label_encoders[col] = LabelEncoder()
    df[col] = label_encoders[col].fit_transform(df[col])

df

,price,maint,doors,persons,lug_capacity,safety,output
0,3,3,0,0,2,1,2
1,3,3,0,0,2,2,2
2,3,3,0,0,2,0,2
3,3,3,0,0,1,1,2
4,3,3,0,0,1,2,2
...,...,...,...,...,...,...,...
1723,1,1,3,2,1,2,1
1724,1,1,3,2,1,0,3
1725,1,1,3,2,0,1,2
1726,1,1,3,2,0,2,1


In [ ]:
# X(특성)와 y(타겟) 분리
X = df.drop('output', axis=1).values
y = df['output'].values
df

,price,maint,doors,persons,lug_capacity,safety,output
0,3,3,0,0,2,1,2
1,3,3,0,0,2,2,2
2,3,3,0,0,2,0,2
3,3,3,0,0,1,1,2
4,3,3,0,0,1,2,2
...,...,...,...,...,...,...,...
1723,1,1,3,2,1,2,1
1724,1,1,3,2,1,0,3
1725,1,1,3,2,0,1,2
1726,1,1,3,2,0,2,1


In [ ]:
# 데이터 표준화 및 학습/테스트 셋 분할
scaler = StandardScaler()
X = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

X_train.shape, X_test.shape, y_train.shape, y_test.shape,

((1382, 6), (346, 6), (1382,), (346,))

In [ ]:
# PyTorch 텐서 및 DataLoader 변환
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.int64)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.int64)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=32, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=32, shuffle=False)

X_train.shape, X_test.shape, y_train.shape, y_test.shape

((1382, 6), (346, 6), (1382,), (346,))

# 4. 분류 모델 정의

In [ ]:
class CarClassifier(nn.Module):
    def __init__(self):
        super(CarClassifier, self).__init__()
        self.fc1 = nn.Linear(6, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 4)  # 출력 클래스 4개

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x

model = CarClassifier()
criterion = nn.CrossEntropyLoss()  # 분류용 손실함수
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 5. 모델학습

In [ ]:
# 5. 모델 학습
epochs = 20
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    # 평가
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    acc = 100 * correct / total
    print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss/len(train_loader):.4f}, Test Accuracy: {acc:.2f}%")

Epoch 1/20, Loss: 0.0094, Test Accuracy: 97.98%
Epoch 2/20, Loss: 0.0089, Test Accuracy: 97.40%
Epoch 3/20, Loss: 0.0090, Test Accuracy: 97.40%
Epoch 4/20, Loss: 0.0109, Test Accuracy: 97.69%
Epoch 5/20, Loss: 0.0104, Test Accuracy: 97.40%
Epoch 6/20, Loss: 0.0100, Test Accuracy: 98.55%
Epoch 7/20, Loss: 0.0093, Test Accuracy: 98.55%
Epoch 8/20, Loss: 0.0095, Test Accuracy: 97.69%
Epoch 9/20, Loss: 0.0072, Test Accuracy: 98.27%
Epoch 10/20, Loss: 0.0066, Test Accuracy: 97.40%
Epoch 11/20, Loss: 0.0071, Test Accuracy: 98.27%
Epoch 12/20, Loss: 0.0075, Test Accuracy: 97.11%
Epoch 13/20, Loss: 0.0066, Test Accuracy: 98.27%
Epoch 14/20, Loss: 0.0060, Test Accuracy: 98.27%
Epoch 15/20, Loss: 0.0057, Test Accuracy: 98.55%
Epoch 16/20, Loss: 0.0053, Test Accuracy: 98.84%
Epoch 17/20, Loss: 0.0054, Test Accuracy: 98.27%
Epoch 18/20, Loss: 0.0056, Test Accuracy: 98.55%
Epoch 19/20, Loss: 0.0049, Test Accuracy: 98.55%
Epoch 20/20, Loss: 0.0047, Test Accuracy: 98.27%
